In [1]:
import sqlite3
import sys  

In [3]:
# Force Python to point to the exact file on your system
# Use the correct absolute Linux path format
db_path = "/home/sachinkumar/Desktop/sponser_proof_gen/twitch_chat.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# This will read from your real log table
cursor.execute("SELECT username, message FROM chat_logs")
rows = cursor.fetchall()

# ... (rest of your embedding math runs here) ...

# This will save chat_emb right next to chat_logs in the exact same file
cursor.execute("DROP TABLE IF EXISTS chat_emb")

In [4]:
# user_query = "SELECT COUNT(*) FROM review_logs"
user_query = """
SELECT message, COUNT(*) AS weight 
FROM chat_logs
GROUP BY message
ORDER BY weight DESC
LIMIT 20
"""

cursor.execute(user_query)

all_rows = cursor.fetchall()

print(f"{'WEIGHT / COUNT':<15} | {'REVIEW MESSAGE'}")
print('-' * 50)
for row in all_rows:
    message = row[0]
    weight = row[1]
    print(f"{weight:<15} | {message}")
    
conn.close()

WEIGHT / COUNT  | REVIEW MESSAGE
--------------------------------------------------
1               | yea it does.
1               | what ? is this working ?
1               | uiag
1               | so the thing is i have to refresh the token generation again and again to make this work
1               | ok ok bye bye
1               | ok bye remember the above thing.
1               | ok bye !!!?? VoHiYo
1               | is this game gud  ?
1               | hi
1               | hello


In [ ]:
import sqlite3
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

# 1. Fetch clean text from the original log table
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT username, message FROM chat_logs")
rows = cursor.fetchall()

dataset = []
for row in rows:
    # Filter out missing strings
    if row[1] and row[1] not in ("Review text not found", "None"):
        dataset.append({"username": row[0], "message": row[1]})

sentences = [item["message"] for item in dataset]
usernames = [item["username"] for item in dataset]

print(f"Loaded {len(sentences)} clean sentences.")

# 2. Compute high-dimensional embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(sentences, show_progress_bar=True)

# 3. Compress vectors to 3D coordinates via PCA
print("Compressing dimensions down to X, Y, Z...")
pca = PCA(n_components=3)
embeddings_3d = pca.fit_transform(embeddings)

# 4. Initialize the new target database table
cursor.execute("DROP TABLE IF EXISTS chat_emb")
cursor.execute("""
CREATE TABLE chat_emb (
    user_name TEXT,
    x REAL,
    y REAL,
    z REAL,
    review TEXT
)
""")

# 5. Fast batch insert into SQLite
print("Writing 3D coordinates to 'chat_emb' table...")
insert_data = []
for i in range(len(dataset)):
    insert_data.append((
        usernames[i],
        float(embeddings_3d[i, 0]),
        float(embeddings_3d[i, 1]),
        float(embeddings_3d[i, 2]),
        sentences[i]
    ))

cursor.executemany(
    "INSERT INTO chat_emb (user_name, x, y, z, review) VALUES (?, ?, ?, ?, ?)",
    insert_data
)
conn.commit()
conn.close()

print("Step 1 complete. 3D spatial data successfully locked into database!")

Loaded 10 clean sentences.


Batches: 100%|██████████| 1/1 [00:00<00:00, 126.92it/s]

Compressing dimensions down to X, Y, Z...
Writing 3D coordinates to 'chat_emb' table...


OperationalError: table chat_emb has no column named review

### Visualizing PCA for better understanding.

In [9]:
import sqlite3
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'browser'

print("Extracting a randomized 5,000 row sample from 'chat_emb'...")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# ORDER BY RANDOM() shuffles the data rows before picking the top 5000
cursor.execute("""
    SELECT user_name, x, y, z, review 
    FROM chat_emb 
    ORDER BY RANDOM() 
    LIMIT 500
""")
rows = cursor.fetchall()
conn.close()

# Unpack columns
usernames = [row[0] for row in rows]
x_coords = [row[1] for row in rows]
y_coords = [row[2] for row in rows]
z_coords = [row[3] for row in rows]
reviews = [row[4] for row in rows]

hover_texts = [msg if len(msg) < 80 else msg[:77] + "..." for msg in reviews]

print(f"Launching browser to render {len(rows)} randomized data points in 3D...")

fig = px.scatter_3d(
    x=x_coords,
    y=y_coords,
    z=z_coords,
    hover_name=reviews,       
    text=usernames,          
    labels={'x': 'PCA X', 'y': 'PCA Y', 'z': 'PCA Z'},
    title="Randomized 3D Semantic Review Interface (5K Sample)"
)

fig.update_traces(
    marker=dict(
        size=3.0,              
        opacity=0.75,           
        line=dict(width=0)     
    ),
    selector=dict(mode='markers')
)

fig.show()

Extracting a randomized 5,000 row sample from 'chat_emb'...
Launching browser to render 10 randomized data points in 3D...


In [10]:
import sqlite3
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'browser'

conn = sqlite3.connect(db_path)
cursor = conn.cursor()
# FIXED: Cleaned up the broken syntax in the SQL text
cursor.execute("""
    SELECT user_name, x, y, z, review 
    FROM chat_emb 
    ORDER BY RANDOM() 
    LIMIT 500
""")
rows = cursor.fetchall()
conn.close()

# Unpack columns
usernames = [row[0] for row in rows]
x_coords = [row[1] for row in rows]
y_coords = [row[2] for row in rows]
z_coords = [row[3] for row in rows]
reviews = [row[4] for row in rows]

# --- TEXT WRAPPING LOGIC ---
formatted_reviews = []
for msg in reviews:
    if not msg:
        formatted_reviews.append("")
        continue
        
    words = msg.split()
    lines = []
    current_line = []
    current_length = 0
    
    for word in words:
        current_line.append(word)
        current_length += len(word) + 1
        
        if current_length >= 50:
            lines.append(" ".join(current_line))
            current_line = []
            current_length = 0
            
    if current_line:
        lines.append(" ".join(current_line))
        
    wrapped_text = "<br>".join(lines)
    formatted_reviews.append(wrapped_text)
# ---------------------------

print(f"Launching browser to render {len(rows)} randomized data points in 3D...")

fig = px.scatter_3d(
    x=x_coords,
    y=y_coords,
    z=z_coords,
    hover_name=usernames,       # FIXED: Shows user_name bolded at the top of the box
    text=formatted_reviews,     # FIXED: Shows the clean multi-line wrapped text inside the box
    labels={'x': 'PCA X', 'y': 'PCA Y', 'z': 'PCA Z'},
    title="Randomized 3D Semantic Chat Interface"
)

fig.update_traces(
    marker=dict(
        size=4.0,              
        opacity=0.80,           
        line=dict(width=0)     
    ),
    selector=dict(mode='markers')
)

fig.show()

Launching browser to render 10 randomized data points in 3D...


In [11]:
import sqlite3
import plotly.express as px
import plotly.io as pio
from sklearn.cluster import KMeans

pio.renderers.default = 'browser'

print("Extracting sample from 'chat_emb'...")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Grab a random sample of reviews
cursor.execute("""
    SELECT user_name, x, y, z, review 
    FROM chat_emb 
    ORDER BY RANDOM() 
    LIMIT 500
""")
rows = cursor.fetchall()
conn.close()

# Unpack columns
usernames = [row[0] for row in rows]
x_coords = [row[1] for row in rows]
y_coords = [row[2] for row in rows]
z_coords = [row[3] for row in rows]
reviews = [row[4] for row in rows]

# --- K-MEANS CLUSTERING LAYER ---
print("Running K-Means clustering...")
# Combine coordinates into a single 2D array for scikit-learn
coordinates = list(zip(x_coords, y_coords, z_coords))

# n_clusters=4 splits the data into 4 distinct colorful groups
# random_state=42 ensures the color groups stay consistent every time you run it
kmeans = KMeans(n_clusters=10, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(coordinates)

# Convert group numbers to string text labels so Plotly treats them as distinct categories
cluster_strings = [f"Cluster {num}" for num in cluster_labels]
# --------------------------------

# --- TEXT WRAPPING LAYER ---
formatted_reviews = []
for msg in reviews:
    if not msg:
        formatted_reviews.append("")
        continue
        
    words = msg.split()
    lines = []
    current_line = []
    current_length = 0
    
    for word in words:
        current_line.append(word)
        current_length += len(word) + 1
        if current_length >= 50:
            lines.append(" ".join(current_line))
            current_line = []
            current_length = 0
            
    if current_line:
        lines.append(" ".join(current_line))
        
    wrapped_text = "<br>".join(lines)
    formatted_reviews.append(wrapped_text)
# ---------------------------

print(f"Launching browser to render clustered data groups...")

fig = px.scatter_3d(
    x=x_coords,
    y=y_coords,
    z=z_coords,
    color=cluster_strings,      # Color-codes the dots dynamically based on K-Means groups
    hover_name=formatted_reviews,       # Changed to usernames (bold title)
    text=usernames,     
    labels={'x': 'PCA X', 'y': 'PCA Y', 'z': 'PCA Z', 'color': 'Group ID'},
    title="K-Means Clustering on 3D Semantic Space (5K Sample)"
)

fig.update_traces(
    marker=dict(
        size=3.0,              
        opacity=0.85,           
        line=dict(width=0)     
    ),
    selector=dict(mode='markers')
)

fig.show()

Extracting sample from 'chat_emb'...
Running K-Means clustering...
Launching browser to render clustered data groups...


Finding Optimal clustering

In [13]:
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

print("Extracting coordinates from 'chat_emb'...")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT x, y, z FROM chat_emb")
rows = cursor.fetchall()
conn.close()

# Convert to a standard NumPy matrix
coordinates = np.array(rows)

# Define the range of cluster counts to test
cluster_range = range(2, 13)
inertia_scores = []

print("Evaluating cluster configurations...")
for k in cluster_range:
    # n_init='auto' keeps the optimization processing fast
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(coordinates)
    inertia_scores.append(kmeans.inertia_)
    print(f"Calculated Inertia for K={k}: {kmeans.inertia_:.2f}")

# Plot the Elbow Curve using standard Matplotlib
plt.figure(figsize=(9, 5))
plt.plot(cluster_range, inertia_scores, marker='o', linestyle='-', color='blue', linewidth=2)
plt.title('The Elbow Method for Optimal K Selection')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Within-Cluster Sum of Squares)')
plt.xticks(cluster_range)
plt.grid(True, linestyle='--', alpha=0.6)

print("\nDisplaying elbow metric chart...")
plt.show()

Extracting coordinates from 'chat_emb'...
Evaluating cluster configurations...
Calculated Inertia for K=2: 2.53
Calculated Inertia for K=3: 1.43
Calculated Inertia for K=4: 0.45
Calculated Inertia for K=5: 0.21
Calculated Inertia for K=6: 0.15
Calculated Inertia for K=7: 0.05
Calculated Inertia for K=8: 0.01
Calculated Inertia for K=9: 0.00
Calculated Inertia for K=10: 0.00


ValueError: n_samples=10 should be >= n_clusters=11.

In [15]:
import sqlite3
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'browser'

print("Extracting a randomized 5,000 row sample from 'chat_emb'...")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# ORDER BY RANDOM() shuffles the data rows before picking the top 5000
cursor.execute("""
    SELECT user_name, x, y, z, review 
    FROM chat_emb 
    ORDER BY RANDOM() 
    LIMIT 500
""")
rows = cursor.fetchall()
conn.close()

# Unpack columns
usernames = [row[0] for row in rows]
x_coords = [row[1] for row in rows]
y_coords = [row[2] for row in rows]
z_coords = [row[3] for row in rows]
reviews = [row[4] for row in rows]

hover_texts = [msg if len(msg) < 80 else msg[:77] + "..." for msg in reviews]

print(f"Launching browser to render {len(rows)} randomized data points in 3D...")

fig = px.scatter_3d(
    x=x_coords,
    y=y_coords,
    z=z_coords,
    hover_name=reviews,       
    text=usernames,          
    labels={'x': 'PCA X', 'y': 'PCA Y', 'z': 'PCA Z'},
    title="Randomized 3D Semantic Review Interface (5K Sample)"
)

fig.update_traces(
    marker=dict(
        size=3.0,              
        opacity=0.75,           
        line=dict(width=0)     
    ),
    selector=dict(mode='markers')
)

fig.show()

Extracting a randomized 5,000 row sample from 'chat_emb'...
Launching browser to render 10 randomized data points in 3D...


In [17]:
import sqlite3
import numpy as np
from sklearn.cluster import KMeans

print("Extracting full dataset coordinates from 'chat_emb'...")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT user_name, x, y, z, review FROM chat_emb")
rows = cursor.fetchall()
conn.close()

# Unpack columns into numpy arrays for fast vector math
usernames = [row[0] for row in rows]
reviews = [row[4] for row in rows]
coordinates = np.array([[row[1], row[2], row[3]] for row in rows])

# 1. Run K-Means on the entire dataset to match your visualization setup
print("Calculating clusters and centroids...")
num_clusters = 8
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(coordinates)

# Extract the calculated X, Y, Z coordinates for the center of each group
centroids = kmeans.cluster_centers_

print("\n=== CLUSTERING GUT-CHECK: EXEGESIS OF CENTROID EXEMPLARS ===")
print("============================================================")

# 2. Loop through each individual cluster group
for cluster_id in range(num_clusters):
    print(f"\n📁 CLUSTER {cluster_id} EXEMPLARS (Closest to Core Meaning):")
    print("-" * 60)
    
    # Get the 3D coordinate profile of the current cluster's center
    centroid = centroids[cluster_id]
    
    # Isolate the index positions of all reviews belonging to this cluster
    cluster_indices = np.where(cluster_labels == cluster_id)[0]
    
    # Extract the actual coordinates for just this cluster's members
    cluster_coords = coordinates[cluster_indices]
    
    # Calculate the Euclidean distance from every member dot to the cluster center
    # Formula: sqrt((x1-x2)^2 + (y1-y2)^2 + (z1-z2)^2)
    distances = np.linalg.norm(cluster_coords - centroid, axis=1)
    
    # Sort the distances and grab the 5 closest indices
    closest_relative_indices = np.argsort(distances)[:5]
    
    # Map back to the absolute indices of our master arrays
    closest_master_indices = cluster_indices[closest_relative_indices]
    
    # 3. Print out the top 5 exemplary reviews
    for rank, idx in enumerate(closest_master_indices, 1):
        user = usernames[idx]
        text = reviews[idx].strip().replace("\n", " ")
        # Truncate text block gracefully for terminal viewing
        display_text = text if len(text) < 120 else text[:117] + "..."
        
        print(f" {rank}. [{user}]: {display_text}")

Extracting full dataset coordinates from 'chat_emb'...
Calculating clusters and centroids...

=== CLUSTERING GUT-CHECK: EXEGESIS OF CENTROID EXEMPLARS ===

📁 CLUSTER 0 EXEMPLARS (Closest to Core Meaning):
------------------------------------------------------------
 1. [lynx_pinata]: ok bye !!!?? VoHiYo

📁 CLUSTER 1 EXEMPLARS (Closest to Core Meaning):
------------------------------------------------------------
 1. [lynx_pinata]: yea it does.

📁 CLUSTER 2 EXEMPLARS (Closest to Core Meaning):
------------------------------------------------------------
 1. [lynx_pinata]: hi
 2. [lynx_pinata]: hello

📁 CLUSTER 3 EXEMPLARS (Closest to Core Meaning):
------------------------------------------------------------
 1. [lynx_pinata]: so the thing is i have to refresh the token generation again and again to make this work

📁 CLUSTER 4 EXEMPLARS (Closest to Core Meaning):
------------------------------------------------------------
 1. [lynx_pinata]: is this game gud  ?

📁 CLUSTER 5 EXEMPLARS (C